# BlueCarbon-AI: training notebook

This notebook runs the whole BlueCarbon-AI pipeline on a Colab GPU:

1. It downloads cloud-free Sentinel-2 composites, elevation, tidal-wetland probability and fused reference labels for 44 coastal sites through Google Earth Engine.
2. It cuts them into tiles with a leakage-free spatial split. Mission Bay, Moreton Bay, Tampa Bay, Plum Island and Shoalwater Bay are held out entirely.
3. It trains a LightGBM spectral model and a U-Net, then keeps the one that maps blue carbon habitats best.
4. It maps every site plus the Mission Bay 2018 → 2024 change case study, and packages everything as `results.zip`.

**Before you start:** go to *Runtime → Change runtime type* and pick a GPU. You also need an Earth Engine-enabled Google Cloud project; the default is `bluecarbon-ai`.

The run is resumable. Downloaded sites are kept in your Google Drive (about 6 GB), so if Colab disconnects, run every cell again from the top and it continues where it stopped.

In [ ]:
%cd /content
!git clone -q https://github.com/yanicksanchez14-creator/bluecarbon-ai.git 2>/dev/null; git -C /content/bluecarbon-ai fetch -q origin && git -C /content/bluecarbon-ai reset -q --hard origin/main
%cd /content/bluecarbon-ai
!git log -1 --format="Code version: %h  %s"
!pip -q install -e ".[gee]"


In [ ]:
import ee
PROJECT = "bluecarbon-ai"   # <- your Earth Engine cloud project
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine OK:", ee.Number(1).getInfo() == 1)


In [ ]:
# Keep downloaded satellite data in Google Drive, so a Colab disconnect never loses it.
# Re-running the training cell after a disconnect then skips every site already downloaded.
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
cache = "/content/drive/MyDrive/bluecarbon-ai-cache/sites"
link = "/content/bluecarbon-ai/runs/default/sites"
os.makedirs(cache, exist_ok=True)
os.makedirs(os.path.dirname(link), exist_ok=True)
if os.path.isdir(link) and not os.path.islink(link):   # sites downloaded earlier in this session
    shutil.copytree(link, cache, dirs_exist_ok=True)
    shutil.rmtree(link)
if not os.path.islink(link):
    os.symlink(cache, link)
done = [d for d in os.listdir(cache) if os.path.exists(os.path.join(cache, d, "meta.json"))]
print(f"{len(done)} sites already saved in Google Drive")


In [ ]:
!python scripts/retrain.py

In [ ]:
from google.colab import files
files.download("results.zip")

## Optional: kelp canopy (can run in a second Colab session)

Run cells 1, 2 and 3 first (code, Earth Engine, Drive cache), then these two. About 1 to 2 hours; resumable. Downloads quarterly Sentinel-2 composites for 7 California kelp beds plus Kelpwatch's public data, calibrates, and writes `results_kelp.zip`.

In [ ]:
!pip -q install xarray netCDF4
!python scripts/kelp_pipeline.py

In [ ]:
from google.colab import files
files.download("results_kelp.zip")